## Import Necessary Libraries

In [198]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from feature_engine.datetime import DatetimeFeatures
from feature_engine.timeseries.forecasting import (
    LagFeatures, 
    WindowFeatures, 
    ExpandingWindowFeatures
)


%matplotlib inline

## Loading Data

In [199]:
data = pd.read_csv('../3_Data/processed/2025_hourly_all_cleaned.csv')
data.head()

,passenger_demand,taxi_demand,timestamp
0,9132,7344,2025-01-01 00:00:00
1,8996,8468,2025-01-01 01:00:00
2,7364,7257,2025-01-01 02:00:00
3,4904,4915,2025-01-01 03:00:00
4,3015,2918,2025-01-01 04:00:00


In [200]:
data

,passenger_demand,taxi_demand,timestamp
0,9132,7344,2025-01-01 00:00:00
1,8996,8468,2025-01-01 01:00:00
2,7364,7257,2025-01-01 02:00:00
3,4904,4915,2025-01-01 03:00:00
4,3015,2918,2025-01-01 04:00:00
...,...,...,...
6547,9595,9779,2025-09-30 19:00:00
6548,8882,9539,2025-09-30 20:00:00
6549,9048,9965,2025-09-30 21:00:00
6550,7026,8001,2025-09-30 22:00:00


In [201]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6552 entries, 0 to 6551
Data columns (total 3 columns):
 #   Column            Non-Null Count  Dtype 
---  ------            --------------  ----- 
 0   passenger_demand  6552 non-null   int64 
 1   taxi_demand       6552 non-null   int64 
 2   timestamp         6552 non-null   object
dtypes: int64(2), object(1)
memory usage: 153.7+ KB


In [202]:
df = data.copy()

In [203]:
df

,passenger_demand,taxi_demand,timestamp
0,9132,7344,2025-01-01 00:00:00
1,8996,8468,2025-01-01 01:00:00
2,7364,7257,2025-01-01 02:00:00
3,4904,4915,2025-01-01 03:00:00
4,3015,2918,2025-01-01 04:00:00
...,...,...,...
6547,9595,9779,2025-09-30 19:00:00
6548,8882,9539,2025-09-30 20:00:00
6549,9048,9965,2025-09-30 21:00:00
6550,7026,8001,2025-09-30 22:00:00


## Add Temporal Features

In [204]:
def add_temporal_features(dataframe: pd.DataFrame, datetime_variable: str = 'timestamp') -> pd.DataFrame:
    """
    Adds temporal features to the dataframe using Feature-engine's DatetimeFeatures.

    Parameters:
    ----------
    dataframe : pd.DataFrame
        Input dataframe containing the datetime column.
    datetime_variable : str
        Name of the datetime column. Default is 'timestamp'.

    Returns:
    -------
    pd.DataFrame
        DataFrame with new temporal features appended.
    """
    # Ensure the datetime column is of datetime type
    dataframe[datetime_variable] = pd.to_datetime(dataframe[datetime_variable])
    
    features_to_extract = [
        "month", "quarter","semester","year","week","day_of_week","day_of_month",
        "day_of_year","weekend","month_start","month_end","quarter_start",
        "quarter_end","year_start","year_end","leap_year","days_in_month","hour","minute","second"
    ]

    # Initialize DatetimeFeatures transformer
    dt_feat = DatetimeFeatures(
        variables=[datetime_variable],
        features_to_extract=features_to_extract
    )
    
    # Fit and transform
    temporal_features = dt_feat.fit_transform(dataframe[[datetime_variable]])
    
    # Merge new features back into the original dataframe
    dataframe = pd.concat([dataframe, temporal_features], axis=1)
    
    return dataframe

In [205]:
### Add Temporal Features
df = add_temporal_features(df, datetime_variable='timestamp')
df.head()

,passenger_demand,taxi_demand,timestamp,timestamp_month,timestamp_quarter,timestamp_semester,timestamp_year,timestamp_week,timestamp_day_of_week,timestamp_day_of_month,...,timestamp_month_end,timestamp_quarter_start,timestamp_quarter_end,timestamp_year_start,timestamp_year_end,timestamp_leap_year,timestamp_days_in_month,timestamp_hour,timestamp_minute,timestamp_second
0,9132,7344,2025-01-01 00:00:00,1,1,1,2025,1,2,1,...,0,1,0,1,0,0,31,0,0,0
1,8996,8468,2025-01-01 01:00:00,1,1,1,2025,1,2,1,...,0,1,0,1,0,0,31,1,0,0
2,7364,7257,2025-01-01 02:00:00,1,1,1,2025,1,2,1,...,0,1,0,1,0,0,31,2,0,0
3,4904,4915,2025-01-01 03:00:00,1,1,1,2025,1,2,1,...,0,1,0,1,0,0,31,3,0,0
4,3015,2918,2025-01-01 04:00:00,1,1,1,2025,1,2,1,...,0,1,0,1,0,0,31,4,0,0


In [206]:
df.head().T

,0,1,2,3,4
passenger_demand,9132,8996,7364,4904,3015
taxi_demand,7344,8468,7257,4915,2918
timestamp,2025-01-01 00:00:00,2025-01-01 01:00:00,2025-01-01 02:00:00,2025-01-01 03:00:00,2025-01-01 04:00:00
timestamp_month,1,1,1,1,1
timestamp_quarter,1,1,1,1,1
timestamp_semester,1,1,1,1,1
timestamp_year,2025,2025,2025,2025,2025
timestamp_week,1,1,1,1,1
timestamp_day_of_week,2,2,2,2,2
timestamp_day_of_month,1,1,1,1,1


In [207]:
df.filter(regex='timestamp_').head()

,timestamp_month,timestamp_quarter,timestamp_semester,timestamp_year,timestamp_week,timestamp_day_of_week,timestamp_day_of_month,timestamp_day_of_year,timestamp_weekend,timestamp_month_start,timestamp_month_end,timestamp_quarter_start,timestamp_quarter_end,timestamp_year_start,timestamp_year_end,timestamp_leap_year,timestamp_days_in_month,timestamp_hour,timestamp_minute,timestamp_second
0,1,1,1,2025,1,2,1,1,0,1,0,1,0,1,0,0,31,0,0,0
1,1,1,1,2025,1,2,1,1,0,1,0,1,0,1,0,0,31,1,0,0
2,1,1,1,2025,1,2,1,1,0,1,0,1,0,1,0,0,31,2,0,0
3,1,1,1,2025,1,2,1,1,0,1,0,1,0,1,0,0,31,3,0,0
4,1,1,1,2025,1,2,1,1,0,1,0,1,0,1,0,0,31,4,0,0


## LagFeatures
<b>Lag:</b> In various fields, "lag" often refers to a delay or a period of time between two events or actions. For instance, in economics or finance, it might represent the delay between a policy change and its effects on the economy. In statistics, a lag is a shift in time for a variable in a time series analysis. It's used to observe the relationship between past and present values of a variable.

In [208]:
def add_lag_features(df: pd.DataFrame) -> pd.DataFrame:
    """
    Adds lag features for 'passenger_demand' and 'taxi_demand'.

    Lag periods: 1, 2, 4, 8, 16, 24 hours
    Missing values are handled gracefully.
    """
    try:
        if df['timestamp'].dtype != 'datetime64[ns]':
            df['timestamp'] = pd.to_datetime(df['timestamp'])

        lag_periods = [1, 2, 4, 8, 16, 24]
        lag_variables = ["passenger_demand", "taxi_demand"]
        
        # Initialize LagFeatures transformer
        lag_transformer = LagFeatures(
            variables=lag_variables,
            periods=lag_periods,
            sort_index=True,
            missing_values='ignore',  # Avoid errors for first few NaNs
            drop_original=False
        )
        
        # Fit & transform
        lag_df = lag_transformer.fit_transform(df[['timestamp'] + lag_variables])
        
         # Append lag columns to original df
        for col in lag_df.columns:
            if col not in df.columns:  # avoid overwriting
                df[col] = lag_df[col].values
        
        print(f"Lag features added: {list(lag_df.columns)}")
        return df
    
    except Exception as e:
        print(f"Error in add_lag_features(): {e}")
        return df

In [209]:
df = add_lag_features(df)
df.head()

Lag features added: ['timestamp', 'passenger_demand', 'taxi_demand', 'passenger_demand_lag_1', 'taxi_demand_lag_1', 'passenger_demand_lag_2', 'taxi_demand_lag_2', 'passenger_demand_lag_4', 'taxi_demand_lag_4', 'passenger_demand_lag_8', 'taxi_demand_lag_8', 'passenger_demand_lag_16', 'taxi_demand_lag_16', 'passenger_demand_lag_24', 'taxi_demand_lag_24']


,passenger_demand,taxi_demand,timestamp,timestamp_month,timestamp_quarter,timestamp_semester,timestamp_year,timestamp_week,timestamp_day_of_week,timestamp_day_of_month,...,passenger_demand_lag_2,taxi_demand_lag_2,passenger_demand_lag_4,taxi_demand_lag_4,passenger_demand_lag_8,taxi_demand_lag_8,passenger_demand_lag_16,taxi_demand_lag_16,passenger_demand_lag_24,taxi_demand_lag_24
0,9132,7344,2025-01-01 00:00:00,1,1,1,2025,1,2,1,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,8996,8468,2025-01-01 01:00:00,1,1,1,2025,1,2,1,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,7364,7257,2025-01-01 02:00:00,1,1,1,2025,1,2,1,...,9132.0,7344.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,4904,4915,2025-01-01 03:00:00,1,1,1,2025,1,2,1,...,8996.0,8468.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,3015,2918,2025-01-01 04:00:00,1,1,1,2025,1,2,1,...,7364.0,7257.0,9132.0,7344.0,NaN,NaN,NaN,NaN,NaN,NaN


In [210]:
df.head().T

,0,1,2,3,4
passenger_demand,9132,8996,7364,4904,3015
taxi_demand,7344,8468,7257,4915,2918
timestamp,2025-01-01 00:00:00,2025-01-01 01:00:00,2025-01-01 02:00:00,2025-01-01 03:00:00,2025-01-01 04:00:00
timestamp_month,1,1,1,1,1
timestamp_quarter,1,1,1,1,1
timestamp_semester,1,1,1,1,1
timestamp_year,2025,2025,2025,2025,2025
timestamp_week,1,1,1,1,1
timestamp_day_of_week,2,2,2,2,2
timestamp_day_of_month,1,1,1,1,1


## WindowFeatures
<b>Windowing in Data Analysis:</b> In data analysis or signal processing, a window function is a mathematical function that modifies a time series or sequence of data to emphasize or de-emphasize certain points. It's often used in techniques like Fourier analysis, signal processing, and data smoothing. The function "windows" or narrows the focus to a specific section of the data, reducing the impact of values outside that section. </br> </br>
Both concepts are crucial in various domains, helping to understand patterns, relationships, and the behavior of data over time.</br> </br>
The mean value of the previous 3 months of data is a window feature. The maximum value of the previous three rows of data is another window feature.

In [211]:
def add_window_features(
    df: pd.DataFrame,
    variables: list = None,
    window: int = 7,
    functions: list = ['mean', 'std', 'median'],
) -> pd.DataFrame:
    """
    Adds rolling window features to the dataframe using Feature-engine's WindowFeatures.

    Parameters:
    - df: Input DataFrame with 'timestamp' column and target variables.
    - variables: List of numeric variables to create window features for. Default: ['passenger_demand', 'taxi_demand'].
    - window: Rolling window size (number of observations). Default: 7.
    - functions: List of aggregation functions. Default: ['mean', 'std', 'median'].

    Returns:
    - df: DataFrame with additional window features.
    """

    try:
        # Ensure timestamp is datetime
        if df['timestamp'].dtype != 'datetime64[ns]':
            df['timestamp'] = pd.to_datetime(df['timestamp'])

        # Default variables
        if variables is None:
            variables = ['passenger_demand', 'taxi_demand']

        # Initialize WindowFeatures transformer
        window_transformer = WindowFeatures(
            variables=variables,
            window=window,
            min_periods=1,  # handle small windows
            functions=functions,
            periods=1,      # lag the window by 1 to avoid lookahead bias
            freq=None,
            sort_index=True,
            missing_values='ignore',  # avoid errors for first few rows
            drop_original=False
        )

        # Fit & transform
        window_df = window_transformer.fit_transform(df[['timestamp'] + variables])

        # Append new window features to original df
        for col in window_df.columns:
            if col not in df.columns:  # avoid overwriting
                df[col] = window_df[col].values

        print(f"Window features added: {list(window_df.columns[1:])}")
        return df

    except Exception as e:
        print(f"Error in add_window_features(): {e}")
        return df


In [212]:
df = add_window_features(df)
df.head()

Window features added: ['passenger_demand', 'taxi_demand', 'passenger_demand_window_7_mean', 'passenger_demand_window_7_std', 'passenger_demand_window_7_median', 'taxi_demand_window_7_mean', 'taxi_demand_window_7_std', 'taxi_demand_window_7_median']


,passenger_demand,taxi_demand,timestamp,timestamp_month,timestamp_quarter,timestamp_semester,timestamp_year,timestamp_week,timestamp_day_of_week,timestamp_day_of_month,...,passenger_demand_lag_16,taxi_demand_lag_16,passenger_demand_lag_24,taxi_demand_lag_24,passenger_demand_window_7_mean,passenger_demand_window_7_std,passenger_demand_window_7_median,taxi_demand_window_7_mean,taxi_demand_window_7_std,taxi_demand_window_7_median
0,9132,7344,2025-01-01 00:00:00,1,1,1,2025,1,2,1,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,8996,8468,2025-01-01 01:00:00,1,1,1,2025,1,2,1,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,7364,7257,2025-01-01 02:00:00,1,1,1,2025,1,2,1,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,4904,4915,2025-01-01 03:00:00,1,1,1,2025,1,2,1,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,3015,2918,2025-01-01 04:00:00,1,1,1,2025,1,2,1,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [213]:
df.head().T

,0,1,2,3,4
passenger_demand,9132,8996,7364,4904,3015
taxi_demand,7344,8468,7257,4915,2918
timestamp,2025-01-01 00:00:00,2025-01-01 01:00:00,2025-01-01 02:00:00,2025-01-01 03:00:00,2025-01-01 04:00:00
timestamp_month,1,1,1,1,1
timestamp_quarter,1,1,1,1,1
timestamp_semester,1,1,1,1,1
timestamp_year,2025,2025,2025,2025,2025
timestamp_week,1,1,1,1,1
timestamp_day_of_week,2,2,2,2,2
timestamp_day_of_month,1,1,1,1,1


## ExpandingWindowFeatures

In [214]:
def add_expanding_window_features(
    df: pd.DataFrame,
    variables: list = None,
    functions: list = ['std'],
    min_periods: int = 1
) -> pd.DataFrame:
    """
    Adds expanding window features to the dataframe using Feature-engine's ExpandingWindowFeatures.

    Parameters:
    - df: Input DataFrame with 'timestamp' column and target variables.
    - variables: List of numeric variables to create expanding window features for. Default: ['passenger_demand', 'taxi_demand'].
    - functions: List of aggregation functions. Default: ['std'].
    - min_periods: Minimum number of observations in expanding window. Default: 1.

    Returns:
    - df: DataFrame with additional expanding window features.
    """

    try:
        # Ensure timestamp is datetime
        if df['timestamp'].dtype != 'datetime64[ns]':
            df['timestamp'] = pd.to_datetime(df['timestamp'])

        # Default variables
        if variables is None:
            variables = ['passenger_demand', 'taxi_demand']

        # Initialize ExpandingWindowFeatures transformer
        exp_transformer = ExpandingWindowFeatures(
            variables=variables,
            min_periods=min_periods,
            functions=functions,
            periods=1,           # lag by 1 to prevent lookahead
            freq=None,
            sort_index=True,
            missing_values='ignore',  # avoid errors for first few rows
            drop_original=False
        )

        # Fit & transform
        exp_df = exp_transformer.fit_transform(df[['timestamp'] + variables])

        # Append new expanding window features to original df
        for col in exp_df.columns:
            if col not in df.columns:  # avoid overwriting
                df[col] = exp_df[col].values

        print(f"Expanding window features added: {list(exp_df.columns[1:])}")
        return df

    except Exception as e:
        print(f"Error in add_expanding_window_features(): {e}")
        return df


In [215]:
df = add_expanding_window_features(df)
df.head()

Expanding window features added: ['passenger_demand', 'taxi_demand', 'passenger_demand_expanding_std', 'taxi_demand_expanding_std']


,passenger_demand,taxi_demand,timestamp,timestamp_month,timestamp_quarter,timestamp_semester,timestamp_year,timestamp_week,timestamp_day_of_week,timestamp_day_of_month,...,passenger_demand_lag_24,taxi_demand_lag_24,passenger_demand_window_7_mean,passenger_demand_window_7_std,passenger_demand_window_7_median,taxi_demand_window_7_mean,taxi_demand_window_7_std,taxi_demand_window_7_median,passenger_demand_expanding_std,taxi_demand_expanding_std
0,9132,7344,2025-01-01 00:00:00,1,1,1,2025,1,2,1,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,8996,8468,2025-01-01 01:00:00,1,1,1,2025,1,2,1,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,7364,7257,2025-01-01 02:00:00,1,1,1,2025,1,2,1,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,96.166522,794.788022
3,4904,4915,2025-01-01 03:00:00,1,1,1,2025,1,2,1,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,983.848227,675.458610
4,3015,2918,2025-01-01 04:00:00,1,1,1,2025,1,2,1,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1968.074186,1492.935587


In [216]:
df.head().T

,0,1,2,3,4
passenger_demand,9132,8996,7364,4904,3015
taxi_demand,7344,8468,7257,4915,2918
timestamp,2025-01-01 00:00:00,2025-01-01 01:00:00,2025-01-01 02:00:00,2025-01-01 03:00:00,2025-01-01 04:00:00
timestamp_month,1,1,1,1,1
timestamp_quarter,1,1,1,1,1
timestamp_semester,1,1,1,1,1
timestamp_year,2025,2025,2025,2025,2025
timestamp_week,1,1,1,1,1
timestamp_day_of_week,2,2,2,2,2
timestamp_day_of_month,1,1,1,1,1


In [217]:
df.dropna(inplace=True)

In [218]:
df.head().T

,24,25,26,27,28
passenger_demand,1344,558,295,303,342
taxi_demand,1051,436,268,220,277
timestamp,2025-01-02 00:00:00,2025-01-02 01:00:00,2025-01-02 02:00:00,2025-01-02 03:00:00,2025-01-02 04:00:00
timestamp_month,1,1,1,1,1
timestamp_quarter,1,1,1,1,1
timestamp_semester,1,1,1,1,1
timestamp_year,2025,2025,2025,2025,2025
timestamp_week,1,1,1,1,1
timestamp_day_of_week,3,3,3,3,3
timestamp_day_of_month,2,2,2,2,2


## Let's Save The FeaturesEngineering Data

In [219]:
df.to_csv('../3_Data/processed/2025_hourly_all_features.csv', index=False)

## Extract Pipeline for Feature Engineering Data